In [ ]:
import pandas as pd
import networkx as nx
import community.community_louvain as community_louvain
from itertools import combinations
from collections import Counter
import os

def perform_optimized_clustering(data_file, mapping_file, target_min=5, target_max=8):
    if not os.path.exists(data_file) or not os.path.exists(mapping_file):
        print("❌ 파일 경로를 확인해주세요.")
        return

    # 1. 매핑 사전 및 데이터 로드 (이전의 전처리 로직 포함)
    df_map = pd.read_csv(mapping_file).dropna(subset=['Keyword', 'Target'])
    mapping_dict = dict(zip(df_map[df_map['Type'] == '영문']['Keyword'].astype(str).str.lower(), 
                            df_map[df_map['Type'] == '영문']['Target'].astype(str)))

    df = pd.read_csv(data_file)
    processed_docs = []
    for k_str in df['키워드'].dropna():
        words = [w.strip().lower() for w in str(k_str).replace(';', ',').split(',') if w.strip()]
        refined = list(set([str(mapping_dict.get(w, w)) for w in words if str(mapping_dict.get(w, w)).lower() != 'nan']))
        if len(refined) >= 2:
            processed_docs.append(refined)

    # 2. 네트워크 생성
    G = nx.Graph()
    edge_counts = Counter()
    for words in processed_docs:
        for pair in combinations(sorted(words), 2):
            edge_counts[pair] += 1
    
    for (n1, n2), w in edge_counts.items():
        if w >= 2: G.add_edge(n1, n2, weight=w)

    # 3. 해상도(Resolution) 자동 조절 루프
    # 군집 개수가 target_min ~ target_max 사이에 들어올 때까지 조절합니다.
    print(f"🔍 적정 군집 개수({target_min}~{target_max}개)를 찾는 중...")
    res = 1.0
    best_partition = None
    
    for attempt in range(20): # 최대 20번 시도
        partition = community_louvain.best_partition(G, weight='weight', resolution=res, random_state=42)
        num_clusters = len(set(partition.values()))
        
        if target_min <= num_clusters <= target_max:
            best_partition = partition
            break
        elif num_clusters < target_min:
            res *= 1.1 # 군집이 너무 적으면 해상도 높임 (더 잘게 쪼갬)
        else:
            res *= 0.9 # 군집이 너무 많으면 해상도 낮춤 (더 크게 뭉침)
    
    if best_partition is None: best_partition = partition # 실패시 마지막 결과 사용

    # 4. 결과 요약 및 라벨링 후보 추출
    clusters = {}
    for node, c_id in best_partition.items():
        clusters.setdefault(c_id, []).append(node)

    summary_data = []
    print(f"\n✅ 최종 {len(clusters)}개의 군집이 생성되었습니다. (해상도: {res:.2f})")
    print("-" * 60)

    for c_id, nodes in sorted(clusters.items()):
        # 중심성(Degree) 기준으로 상위 키워드 10개 추출
        top_k = sorted(nodes, key=lambda x: G.degree(x), reverse=True)[:10]
        nodes_str = ", ".join(top_k)
        print(f"군집 {c_id+1}: {nodes_str}")
        
        summary_data.append({
            'Cluster_ID': c_id + 1,
            'Keywords_Top10': nodes_str,
            'Node_Count': len(nodes),
            'Suggested_Label': '' # 여기에 나중에 라벨을 적으시면 됩니다.
        })

    # 5. 시계열 분석을 위해 원본 데이터에 군집 ID 매핑 및 저장
    # (각 논문의 첫 번째 유효 키워드를 기준으로 군집 할당 - 단순화된 매핑)
    # 실제로는 논문별 Cluster ID를 결정하는 별도의 로직이 필요할 수 있습니다.
    
    df_summary = pd.DataFrame(summary_data)
    df_summary.to_csv('군집별_라벨링_작업표.csv', index=False, encoding='utf-8-sig')
    print("-" * 60)
    print("📂 '군집별_라벨링_작업표.csv'가 생성되었습니다. 이 파일의 Suggested_Label 칸을 채워보세요.")

    return G, best_partition, df_summary

if __name__ == "__main__":
    G, partition, summary = perform_optimized_clustering('법학_AI_논문_상세정보_리스트.csv', '키워드 변환 표.csv')